HGA

In [8]:
from blf import blf, scale_factor
import numpy as np
from shapely.strtree import STRtree
from shapely.ops import unary_union
from decimal import Decimal, getcontext
from shapely.geometry import Polygon
from shapely import affinity, touches
from joblib import Parallel, delayed

# simulated annealing algorithm
from numpy import exp
import random
# scale_factor = Decimal('1e18') # be careful with this

# def generate_random_solution(n_trees = 10):
#     tree_order = list(range(1, n_trees + 1))
#     np.random.shuffle(tree_order)
#     return [tree_order, [np.random.uniform(0, 360) for _ in range(n_trees)]]

def generate_random_solution(n_trees = 10):
    return [np.random.uniform(0, 360) for _ in range(n_trees)]

def generate_random_population(population_size = 100, n_trees = 10):
    return [generate_random_solution(n_trees) for _ in range(1, population_size + 1)]

def get_fitness(solution_decoded_trees):
    polygons = [x.polygon for x in solution_decoded_trees]
    polygons_tree = STRtree(polygons)
    num_trees = len(solution_decoded_trees)

    # Checking for collisions
    limit = 100 * scale_factor
    for i, tree in enumerate(solution_decoded_trees):
        if tree.center_x < -limit or tree.center_x > limit or \
           tree.center_y < -limit or tree.center_y > limit:
            return np.inf
        poly = tree.polygon
        indices = polygons_tree.query(poly)
        for index in indices:
            if index == i:  # don't check against self
                continue
            if poly.intersects(polygons[index]) and not poly.touches(polygons[index]):
                return np.inf

    # Calculate score for the group
    bounds = unary_union(polygons).bounds
    # Use the largest edge of the bounding rectangle to make a square boulding box
    side_length_scaled = max(bounds[2] - bounds[0], bounds[3] - bounds[1])

    group_score = (Decimal(side_length_scaled) ** 2) / (scale_factor**2) / Decimal(num_trees)
    return float(group_score)

In [9]:
class ChristmasTree:
    """Represents a single, rotatable Christmas tree of a fixed size."""

    def __init__(self, center_x='0', center_y='0', angle='0'):
        """Initializes the Christmas tree with a specific position and rotation."""
        self.center_x = Decimal(center_x)
        self.center_y = Decimal(center_y)
        self.angle = Decimal(angle)

        trunk_w = Decimal('0.15')
        trunk_h = Decimal('0.2')
        base_w = Decimal('0.7')
        mid_w = Decimal('0.4')
        top_w = Decimal('0.25')
        tip_y = Decimal('0.8')
        tier_1_y = Decimal('0.5')
        tier_2_y = Decimal('0.25')
        base_y = Decimal('0.0')
        trunk_bottom_y = -trunk_h

        initial_polygon = Polygon(
            [
                # Start at Tip
                (Decimal('0.0') * scale_factor, tip_y * scale_factor),
                # Right side - Top Tier
                (top_w / Decimal('2') * scale_factor, tier_1_y * scale_factor),
                (top_w / Decimal('4') * scale_factor, tier_1_y * scale_factor),
                # Right side - Middle Tier
                (mid_w / Decimal('2') * scale_factor, tier_2_y * scale_factor),
                (mid_w / Decimal('4') * scale_factor, tier_2_y * scale_factor),
                # Right side - Bottom Tier
                (base_w / Decimal('2') * scale_factor, base_y * scale_factor),
                # Right Trunk
                (trunk_w / Decimal('2') * scale_factor, base_y * scale_factor),
                (trunk_w / Decimal('2') * scale_factor, trunk_bottom_y * scale_factor),
                # Left Trunk
                (-(trunk_w / Decimal('2')) * scale_factor, trunk_bottom_y * scale_factor),
                (-(trunk_w / Decimal('2')) * scale_factor, base_y * scale_factor),
                # Left side - Bottom Tier
                (-(base_w / Decimal('2')) * scale_factor, base_y * scale_factor),
                # Left side - Middle Tier
                (-(mid_w / Decimal('4')) * scale_factor, tier_2_y * scale_factor),
                (-(mid_w / Decimal('2')) * scale_factor, tier_2_y * scale_factor),
                # Left side - Top Tier
                (-(top_w / Decimal('4')) * scale_factor, tier_1_y * scale_factor),
                (-(top_w / Decimal('2')) * scale_factor, tier_1_y * scale_factor),
            ]
        )
        rotated = affinity.rotate(initial_polygon, float(self.angle), origin=(0, 0))
        self.polygon = affinity.translate(rotated,
                                          xoff=float(self.center_x * scale_factor),
                                          yoff=float(self.center_y * scale_factor))

In [10]:
def aux_NWOX(parent1,parent2,i,j): # Non-Wrapping Ordered Crossover (auxiliary function)
    child = [parent1[x] if i<=x<=j else None for x in range(len(parent1))] # Copying the block of one parent
    search_ind = 0
    for e2 in parent2: # Adding the rest of the elements in the order of the other parent
        if e2 not in child:
            while search_ind < len(child) and child[search_ind] is not None:
                search_ind += 1
            child[search_ind] = e2
    return child

def NWOX(parent1,parent2): # Non-Wrapping Ordered Crossover
    i = random.randint(0,len(parent1)-1)
    j = random.randint(0,len(parent1)-1)
    if j < i:
        i,j = j,i
    child1 = aux_NWOX(parent1,parent2,i,j)
    child2 = aux_NWOX(parent2,parent1,i,j)
    return [child1,child2]

def aux_CX(parent1,parent2): # Cycle Crossover (auxiliary function)
    parents = [parent1,parent2]
    N = len(parent1)
    child= [-1 for x in range(N)]
    end_cycle = True
    while sum(parent1) != sum(child):
        if end_cycle:
            current_index = child.index(-1)
            end_cycle = False
        else:
            posibles = [parent1[current_index],parent2[current_index]]
            choice = random.choice(posibles)
            selected_index_parent = posibles.index(choice)
            if choice in child:
                for i,p in enumerate(posibles):
                    if i != selected_index_parent:
                        if p in child:
                            end_cycle=True
                        else:
                            choice=p
                            selected_index_parent = i
                        break
            other_index_parent = abs(1-selected_index_parent)
            if choice not in child:
                child[current_index] = choice
                current_index = parents[other_index_parent].index(choice)
    return child

def CX(parent1,parent2): # Cycle Crossover
    return [aux_CX(parent1,parent2),aux_CX(parent2,parent1)]

def CX2_modded(parent1,parent2): # Cycle Crossover 2 corrected
    child1,child2 = [],[]
    original_parent1 = list(parent1)
    while len(child1)!=len(original_parent1):
        it = 0
        while True:
            if it==0:
                child1.append(parent2[0])
                it+=1
            else:
                child1.append(new_valueoff1)
            ref1p1 = parent1.index(child1[-1])
            ref2p1 = parent1.index(parent2[ref1p1])
            new_valueoff2 = parent2[ref2p1]
            child2.append(new_valueoff2)
            new_valueoff1 = parent2[parent1.index(new_valueoff2)]
            if new_valueoff1 in child1:
                break
        common = set(child1).intersection(child2)
        if len(common) != len(child1):
            child1 = child1 + [x for x in parent2 if x not in child1]
            child2 = child2 + [x for x in parent1 if x not in child2]
            break
        else:
            parent1 = [x for x in parent1 if x not in common]
            parent2 = [x for x in parent2 if x not in common]
    return [child1,child2]


def arithmetic_crossover(parent1_angles, parent2_angles):
    alpha = random.random()
    child1_angles = [alpha * a1 + (1 - alpha) * a2 for a1, a2 in zip(parent1_angles, parent2_angles)]
    child2_angles = [alpha * a2 + (1 - alpha) * a1 for a1, a2 in zip(parent1_angles, parent2_angles)]
    return [child1_angles, child2_angles]


def simple_swap(child): # Simple Swap (SS)
    i = random.randint(0,len(child)-1)
    j = random.randint(0,len(child)-1)
    child[i],child[j] = child[j],child[i]
    return child

def RSM(child): #Reverse Sequence Mutation: https://arxiv.org/ftp/arxiv/papers/1203/1203.3099.pdf#:~:text=In%20the%20reverse%20sequence%20mutation,covered%20in%20the%20previous%20operation.
    child = list(child) #returning a copy
    i = random.randint(0, len(child)-1)
    j = random.randint(0, len(child)-1)
    if j < i:
        i,j=j,i
    while i < j:
        child[i], child[j] = child[j], child[i]
        i += 1
        j -= 1
    return child

def aa_mutation(child, p_mutation_gene):
    child = list(child)
    for i in range(len(child)):
        r = random.random()
        if r <= p_mutation_gene:
            mutation_angle = random.uniform(-30, 30)  # Mutate angle by up to ±30 degrees
            child[i] = (child[i] + mutation_angle) % 360  # Ensure angle stays within [0, 360)
    return child

def bitwise_mutation(child,p_mut=0.5): # Bitwise mutation
    child = list(child)
    for i in range(len(child)):
        r = random.random()
        if r <= p_mut:
            child[i] = abs(child[i]-1)
    return child


def ranked_wheel_selection(population, decoded_population, fitnesses, sp, parent_number):
    sorted_pop = sorted(population, key = lambda x: fitnesses[population.index(x)], reverse = True)
    sorted_decoded_pop = sorted(decoded_population, key = lambda x: fitnesses[decoded_population.index(x)], reverse = True)
    sorted_fitnesses = sorted(fitnesses, reverse = True)
    sorted_indices = list(range(len(population)))
    ranks = np.array(list(range(1,len(population)+1)))
    scaled_ranks = 2-sp + (2*(sp-1)*(ranks-1)/(ranks.size-1))
    selection_probs = scaled_ranks / np.sum(scaled_ranks)
    selected_indices = np.random.choice(sorted_indices,size=parent_number,p=selection_probs)
    new_pop = [sorted_pop[x] for x in selected_indices]
    new_sorted_decoded_pop = [sorted_decoded_pop[x] for x in selected_indices]
    new_fitnesses = [sorted_fitnesses[x] for x in selected_indices]
    return new_pop, new_sorted_decoded_pop, new_fitnesses


def elitist_replacement(population, decoded_population, population_fitnesses, children, decoded_children, children_fitnesses):
    new_population = population + children
    new_decoded_population = decoded_population + decoded_children
    new_fitnesses = population_fitnesses + children_fitnesses
    new_population = sorted(new_population, key = lambda x: new_fitnesses[new_population.index(x)])
    new_decoded_population = sorted(new_decoded_population, key = lambda x: new_fitnesses[new_decoded_population.index(x)])
    new_fitnesses = sorted(new_fitnesses)
    return new_population[0:len(population)], new_decoded_population[0:len(population)], new_fitnesses[0:len(population)]

Transformations

Slow one

In [11]:
def adjust_solution_blf(solution):
    # indices = sorted(range(len(solution[0])), key=lambda k: solution[0][k])
    # sorted_a = [[solution[0][i] for i in indices], [solution[1][i] for i in indices]]
    # trees = [ChristmasTree(angle=angle) for angle in sorted_a[1]]
    decoded_trees = [ChristmasTree(angle=angle) for _, _, angle in solution]
    placed_trees = blf(decoded_trees)
    # sort back to original order
    # unordered_trees  = [None for _ in range(len(placed_trees))]
    # for idx, original_idx in enumerate(indices):
    #     unordered_trees[original_idx] = placed_trees[idx]
    # return unordered_trees
    return placed_trees

Fast one

In [12]:
def adjust_solution(solution):
    return [ChristmasTree(center_x = center_x, center_y = center_y, angle=angle) for center_x, center_y, angle in solution]

In [43]:
def perturb_gaussian(individual, sigma=5.0):
    new_individual = individual.copy()
    for chromosome in new_individual:
        idx = random.randrange(len(chromosome))
        # Add noise and keep within 0-360 using modulo
        chromosome[idx] = (chromosome[idx] + random.gauss(0, sigma)) % 360
    return new_individual

def simulated_annealing(best, n_iterations = 60, temp = 10):
    best_decoded = adjust_solution(best)
    best_eval = get_fitness(best_decoded)
    curr, curr_eval = best, best_eval
    # run the algorithm
    for i in range(n_iterations):
        # take a step
        candidate = perturb_gaussian(curr)
        candidate_decoded = adjust_solution(candidate)
        candidate_eval = get_fitness(candidate_decoded)
        # check for new best solution
        if candidate_eval < best_eval:
            # store new best point
            best, best_decoded, best_eval = candidate, candidate_decoded, candidate_eval
            # report progress
            # print('>%d f(%s) = %.5f' % (i, best, best_eval))
        # difference between candidate and current point evaluation
        diff = candidate_eval - curr_eval
        # calculate temperature for current epoch
        t = temp / float(i + 1)
        # calculate metropolis acceptance criterion
        r, metropolis = random.random(), exp(-diff / t)
        # check if we should keep the new point
        if diff < 0 or r < metropolis:
            # store the new current point
            curr, curr_eval = candidate, candidate_eval
    return [best, best_decoded, best_eval]

In [14]:
import time
import random
import pandas as pd

In [49]:
def execute(n_trees = 10, configuration=None):
    if configuration is None:
        configuration = ('NWOX', 0.9, 'InheritMask', 'SS', 0.4, 'DivisionSelect', 'RWS-sp-1.5', 'EL', 4)

    # cross_name = configuration[0]
    p_cross = configuration[1]
    # mut_name = configuration[3]
    p_mut = configuration[4]
    # mask_mut = configuration[5]
    sel_name = configuration[6]
    rep_name = configuration[7]
    pop_size = configuration[8]

    final_variables = ["best_fitness", "iterations", "final_time", "final_population"]
    iteration_data = []
    ini_time = time.time()
    iterations = 0
    generated_children = pop_size // 2

    # if cross_name=="NWOX":
    #     crossover = NWOX
    # elif cross_name=="CX":
    #     crossover = CX
    # elif cross_name=="CX2":
    #     crossover = CX2_modded
    # if mut_name=="SS":
    #     mutation = simple_swap
    # elif mut_name=="RSM":
    #     mutation = RSM

    if "RWS" in sel_name:
        selection = ranked_wheel_selection
    if rep_name=="EL":
        replacement = elitist_replacement

    population_blf = generate_random_population(population_size = pop_size, n_trees=n_trees)
    population = [[ [0, 0, angle] for angle in individual] for individual in population_blf]


    decoded_population = [adjust_solution_blf(ind) for ind in population]

    population = [[ [float(tree.center_x / scale_factor),
                     float(tree.center_y / scale_factor),
                     float(tree.angle)
                     ] for tree in ind] for ind in decoded_population]

    # decoded_population = Parallel(n_jobs=-1)(delayed(adjust_solution)(ind) for ind in population)
    fitnesses = [get_fitness(x) for x in decoded_population]

    best_fitness_individual_iteration = min(fitnesses)
    best_individual_iteration = population[fitnesses.index(best_fitness_individual_iteration)]
    iteration_data.append([best_fitness_individual_iteration,0,0,"pop"])
    best_last_fitness = best_fitness_individual_iteration
    repeated_iterations = 0

    while iterations < 10000:
        print(f"Iteration {iterations}, Best fitness: {best_fitness_individual_iteration}")
        ini_iteration_time = time.time()
        parents, _, _ = selection(
            population, decoded_population, fitnesses, 1.5, generated_children
        )

        random_indices = list(range(len(parents)))
        random.shuffle(random_indices)
        parents = [parents[i] for i in random_indices]

        all_children = []
        for i in range(0, generated_children - 1, 2):
            parent_1, parent_2 = parents[i], parents[i+1]
            r = random.random()
            if r <= p_cross:
                parent_1_parts, parent_2_parts = [], []
                for i in range(len(parent_1[0])):
                    parent_1_1, parent_1_2 = [parent_1[j][i] for j in range(len(parent_1))], [parent_2[j][i] for j in range(len(parent_2))]
                    parent_1_1, parent_1_2 = arithmetic_crossover(parent_1_1, parent_1_2)
                    parent_1_parts.append(parent_1_1)
                    parent_2_parts.append(parent_1_2)
                children = [
                    [
                        [max(min(parent_1_parts[0][i], 100), -100),
                        max(min(parent_1_parts[1][i], 100), -100),
                        parent_1_parts[2][i] % 360]
                    for i in range(len(parent_1_parts))
                    ],
                    [
                        [max(min(parent_2_parts[0][i], 100), -100),
                        max(min(parent_2_parts[1][i], 100), -100),
                        parent_2_parts[2][i] % 360]
                    for i in range(len(parent_2_parts))
                ]
                ]
            else:
                children = [parent_1, parent_2]

            mutated_children = []
            decoded_children = []
            children_fitnesses = []
            for i in range(len(children)):
                mutated_child = children[i]
                new_mutated_child = []
                # for chromosome in mutated_child:
                #     r = random.random()
                #     if r <= p_mut:
                #         # new_mutated_child.append(aa_mutation(chromosome, p_mutation_gene=0.25))
                #         new_mutated_child.append(
                #             simulated_annealing(generate_random_solution(n_trees=10), n_iterations = 100, temp = 10)[0]
                #         )
                #     else:
                #         new_mutated_child.append(chromosome)
                r = random.random()
                if r <= p_mut:
                # new_mutated_child.append(aa_mutation(chromosome, p_mutation_gene=0.25))
                    new_mutated_child = simulated_annealing(mutated_child, n_iterations = 100, temp = 10)[0]
                else:
                    new_mutated_child = mutated_child

                mutated_children.append(new_mutated_child)
            for child in children + mutated_children:
                all_children.append(child)

        decoded_children = [adjust_solution(ind) for ind in all_children]
        # decoded_children = Parallel(n_jobs=-1)(delayed(adjust_solution)(ind) for ind in all_children)
        children_fitnesses = [get_fitness(x) for x in decoded_children]

        population, decoded_population, fitnesses = elitist_replacement(
            population, decoded_population, fitnesses, children, decoded_children, children_fitnesses
            )
        
        decoded_population = [adjust_solution_blf(ind) for ind in population]
        population = [[ [float(tree.center_x / scale_factor),
                         float(tree.center_y / scale_factor),
                         float(tree.angle)
                         ] for tree in ind] for ind in decoded_population]
        fitnesses = [get_fitness(x) for x in decoded_population]

        best_individual_iteration = population[0]
        best_fitness_individual_iteration = fitnesses[0]
        iterations+=1
        end_iteration_time = time.time()-ini_iteration_time
        iteration_data.append([best_fitness_individual_iteration, iterations, end_iteration_time, "pop"])

        if best_fitness_individual_iteration >= best_last_fitness:
            repeated_iterations += 1
        else:
            repeated_iterations = 0
        best_last_fitness = best_fitness_individual_iteration

        if iterations==10000 or repeated_iterations == 300: 
            final_time = time.time()-ini_time
            result_data = [[best_fitness_individual_iteration, iterations, final_time, population]]
            result_data = pd.DataFrame(result_data,columns=final_variables)
            iteration_data = pd.DataFrame(iteration_data,columns=final_variables)
            iteration_data.drop(columns=["final_population"],inplace=True)
            return best_individual_iteration, result_data, iteration_data

In [ ]:
results = {}
results_fitness = []
for i in range(1, 201):
    t1 = time.time()
    print(f"Execution {i}")
    best_individual, best_individual_decoded, result_data, iteration_data =\
        best_individual, result_data, iteration_data = execute(n_trees = i)
    results[i] = best_individual_decoded
    results_fitness.append(result_data["best_fitness"][0])
    t2 = time.time()
    print(f"Execution {i} finished in {round(t2 - t1, 2)} seconds with fitness {result_data['best_fitness'][0]}")


def generate_submission_file(tree_problem_solutions: dict):
    # index = [f'{n:03d}_{t}' for n in range(1, 201) for t in range(n)]
    data = []
    for problem_id, tree_list in tree_problem_solutions.items():
        for n, tree in enumerate(tree_list):
            submission_id = f'{problem_id:03d}_{n}'
            data.append([submission_id,
                         f"s{tree.center_x / scale_factor}",
                         f"s{tree.center_y / scale_factor}",
                         f"s{tree.angle}"
            ])
    data = pd.DataFrame(data, columns = ["id", "x", "y", "deg"])
    return data

submission_df = generate_submission_file(results)
submission_df.to_csv("submission_hga_partitioned.csv", index=False)
